# เทรนโมเดล Prophet พยากรณ์ราคาทองคำ (GC=F)
ข้อมูล: `gold_15y.csv` (ราคาปิดรายวันย้อนหลัง 15 ปีจาก Yahoo Finance)

ขั้นตอน: ติดตั้ง → โหลดข้อมูล → ทำความสะอาด → ดูข้อมูล → แบ่ง Train/Test → เทรน → วัดความแม่นยำ (MAE, RMSE, MAPE) เทียบกับ Baseline → Cross-validation หลายช่วงเวลา → เทรนใหม่ด้วยข้อมูลทั้งหมด → พยากรณ์อนาคต → เซฟโมเดล

**กดรันทีละ cell จากบนลงล่าง** (Shift + Enter)

## 1. ติดตั้งไลบรารี (รันครั้งแรกครั้งเดียว)

In [ ]:
%pip install -q prophet pandas numpy matplotlib scikit-learn

## 2. Import และตั้งค่า

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from prophet import Prophet
from prophet.serialize import model_to_json, model_from_json
from sklearn.metrics import mean_absolute_error, mean_squared_error
import logging
logging.getLogger("cmdstanpy").disabled = True   # ปิดข้อความ log ตอนเทรน

CSV_PATH    = "gold_15y.csv"                 # ไฟล์ข้อมูล (วางไว้โฟลเดอร์เดียวกับ notebook)
MODEL_DIR   = "MODEL"                       # โฟลเดอร์เก็บโมเดล
MODEL_PATH  = os.path.join(MODEL_DIR, "gold_prophet_model.json")
FORECAST_PATH = os.path.join(MODEL_DIR, "gold_forecast.csv")
HORIZON     = 7                             # พยากรณ์ล่วงหน้า 7 วันทำการ (ตามขอบเขตข้อ 2.3.6.2)
TEST_DAYS   = HORIZON                       # ชุดทดสอบยาวเท่าระยะที่จะพยากรณ์จริง
FUTURE_DAYS = HORIZON                       # พยากรณ์อนาคตล่วงหน้ากี่วัน
ANCHOR      = True                          # เลื่อนเส้นพยากรณ์ให้เริ่มจากราคาจริงล่าสุด (ลด error ได้มาก)
TREND_DAMP  = 0.5                           # ลดความชันเทรนด์เหลือครึ่งหนึ่ง (Prophet มักลากเทรนด์แรงเกินจริง) 1 = ไม่ลด
LOG_Y       = True                          # เทรนบน log(ราคา): ราคาทองเปลี่ยนเป็น % มากกว่าเป็นดอลลาร์ และพยากรณ์จะไม่ติดลบ

os.makedirs(MODEL_DIR, exist_ok=True)
plt.rcParams["figure.figsize"] = (14, 5)

## 3. โหลดข้อมูล
> ถ้าใช้ **Google Colab** ให้รัน cell ข้างล่างนี้ก่อน เพื่ออัปโหลดไฟล์ CSV (ถ้ารันบนเครื่องตัวเองข้ามได้)

In [ ]:
# เฉพาะ Google Colab: อัปโหลดไฟล์ CSV
try:
    from google.colab import files
    files.upload()
except ImportError:
    print("ไม่ได้รันบน Colab — ข้าม cell นี้ได้")

In [ ]:
raw = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
print("จำนวนแถว:", len(raw))
raw.head()

In [ ]:
raw.info()
raw.describe()

## 4. ทำความสะอาดข้อมูล

In [ ]:
df = raw.copy()

# 4.1 แปลงชนิดข้อมูล
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
for c in ["Open", "High", "Low", "Close", "Volume"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# 4.2 ตัดคอลัมน์ที่ไม่ใช้ (Adj Close เท่ากับ Close สำหรับ futures)
df = df.drop(columns=["Adj Close"], errors="ignore")

# 4.3 ตัดแถวที่วันที่/ราคาปิดว่าง และวันที่ซ้ำ แล้วเรียงตามวันที่
before = len(df)
df = df.dropna(subset=["Date", "Close"])
df = df.drop_duplicates(subset="Date", keep="last").sort_values("Date")
print("ตัดแถวว่าง/ซ้ำ:", before - len(df))

# 4.4 ตัดแถวของวันนี้ (ตลาดยังไม่ปิด ราคายังไม่ใช่ราคาปิดจริง)
today = pd.Timestamp.today().normalize()
print("ตัดแถววันนี้ (ยังไม่ปิดตลาด):", (df["Date"] >= today).sum())
df = df[df["Date"] < today]

# 4.5 ปัดทศนิยมเป็น 2 ตำแหน่ง (แก้ค่าเพี้ยนอย่าง 4216.2002)
df[["Open", "High", "Low", "Close"]] = df[["Open", "High", "Low", "Close"]].round(2)

# 4.7 ตรวจหาค่าผิดปกติ: ราคาเปลี่ยนเกิน 15% ในวันเดียว (แสดงให้ดูเฉย ๆ ไม่ได้ตัดทิ้ง)
ret = df["Close"].pct_change()
print("\nวันที่ราคาเปลี่ยน > 15%:")
print(df.assign(change_pct=(ret * 100).round(2))[ret.abs() > 0.15][["Date", "Close", "change_pct"]])

df = df.reset_index(drop=True)
print("\nเหลือ", len(df), "แถว ตั้งแต่", df["Date"].min().date(), "ถึง", df["Date"].max().date())
print("ค่าว่างที่เหลือ:\n", df.isna().sum())

## 5. เตรียมข้อมูลให้ Prophet (ต้องมีคอลัมน์ `ds` และ `y`)

In [ ]:
data = df[["Date", "Close"]].rename(columns={"Date": "ds", "Close": "y"})
data.tail()

In [ ]:
plt.plot(data["ds"], data["y"])
plt.title("Gold Futures (GC=F) Close Price")
plt.xlabel("Date"); plt.ylabel("Close (USD)")
plt.grid(alpha=.3); plt.show()

## 6. แบ่งข้อมูล Train / Test
ใช้ข้อมูลช่วงท้ายยาว `TEST_DAYS` วัน (= `HORIZON`) เป็นชุดทดสอบ ที่เหลือใช้เทรน (แบ่งตามเวลา ห้ามสุ่ม)

In [ ]:
cutoff = data["ds"].max() - pd.Timedelta(days=TEST_DAYS)
train = data[data["ds"] <= cutoff]
test  = data[data["ds"] >  cutoff]
print(f"Train: {len(train)} แถว  ({train.ds.min().date()} ถึง {train.ds.max().date()})")
print(f"Test : {len(test)} แถว  ({test.ds.min().date()} ถึง {test.ds.max().date()})")

plt.plot(train["ds"], train["y"], label="Train")
plt.plot(test["ds"], test["y"], label="Test", color="orange")
plt.legend(); plt.grid(alpha=.3); plt.show()

## 7. เทรนโมเดล Prophet
เทคนิคที่ใช้ลด error:
- **log(ราคา)**: `fit_model` แปลงราคาเป็น log ก่อนเทรน แล้ว `predict_price` แปลงกลับเป็นดอลลาร์
- **Anchoring**: Prophet มักพยากรณ์ "ระดับราคา" คลาดจากราคาล่าสุด จึงเลื่อนทั้งเส้นพยากรณ์ให้เริ่มจากราคาปิดจริงวันสุดท้าย โดยยังใช้ทิศทางเทรนด์ของ Prophet
- **Trend damping** (`TREND_DAMP=0.5`): ใช้ทิศทางเทรนด์ของ Prophet แค่ครึ่งเดียว เพราะ Prophet มักลากเทรนด์ล่าสุดแรงเกินจริง
- ไม่ใช้ฤดูกาลรายปี ค่าทั้งหมดนี้เลือกจาก cross-validation หลายรอบ ไม่ได้เลือกให้ตรงกับชุดทดสอบชุดเดียว

In [ ]:
def make_model():
    return Prophet(
        daily_seasonality=False,        # ข้อมูลรายวัน ไม่มีรอบภายในวัน
        weekly_seasonality=False,       # ตลาดปิดเสาร์-อาทิตย์ รอบสัปดาห์ไม่มีความหมาย
        yearly_seasonality=False,      # ทดสอบแล้วปิดฤดูกาลรายปีให้ผลดีกว่า
        changepoint_prior_scale=0.05,   # ความยืดหยุ่นของเทรนด์ (เลือกจาก cross-validation)
        changepoint_range=0.95,         # ให้เทรนด์เปลี่ยนได้ถึง 95% ของข้อมูล (ค่าเริ่มต้น 0.8 ทำให้โมเดลไม่เห็นเทรนด์ 3 ปีล่าสุด)
    )

def fit_model(df):
    d = df[["ds", "y"]].copy()
    if LOG_Y:
        d["y"] = np.log(d["y"])
    m = make_model()
    m.fit(d)
    return m

def predict_price(m, ds_df):
    p = m.predict(ds_df)
    cols = ["yhat", "yhat_lower", "yhat_upper"]
    if ANCHOR:   # ให้เส้นพยากรณ์เริ่มที่ราคาจริงวันสุดท้ายที่โมเดลเห็น และลดความชันเทรนด์ตาม TREND_DAMP
        last = m.history.iloc[[-1]]
        base = m.predict(last[["ds"]])["yhat"].iloc[0]
        lo_w, hi_w = p["yhat"] - p["yhat_lower"], p["yhat_upper"] - p["yhat"]   # คงความกว้างช่วงความเชื่อมั่นไว้
        p["yhat"] = last["y"].iloc[0] + TREND_DAMP * (p["yhat"] - base)
        p["yhat_lower"], p["yhat_upper"] = p["yhat"] - lo_w, p["yhat"] + hi_w
    if LOG_Y:
        p[cols] = np.exp(p[cols])
    return p

model = fit_model(train)

## 8. พยากรณ์ช่วง Test

In [ ]:
pred = predict_price(model, test[["ds"]])
result = test.merge(pred[["ds", "yhat", "yhat_lower", "yhat_upper"]], on="ds")
result.head()

## 9. วัดความแม่นยำ: MAE, RMSE, MAPE
- **MAE** ผิดเฉลี่ยกี่ดอลลาร์
- **RMSE** เหมือน MAE แต่ลงโทษวันที่ผิดมาก ๆ หนักกว่า
- **MAPE** ผิดเฉลี่ยกี่ % (ยิ่งต่ำยิ่งดี: <10% ดีมาก, 10–20% ดี, 20–50% พอใช้, >50% ไม่แม่น)
- **Accuracy** ความแม่นยำ = 100 − MAPE (ยิ่งสูงยิ่งดี เช่น MAPE 5% = แม่นยำ 95%)

**Baseline (Naive)** = เดาว่าราคาทุกวันในอนาคตเท่ากับราคาปิดวันสุดท้ายของ Train ถ้าโมเดลแพ้ Baseline แปลว่าโมเดลยังไม่ได้ช่วยอะไร

In [ ]:
def evaluate(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return {"MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
            "MAPE (%)": mape,
            "Accuracy (%)": 100 - mape}

y_true = result["y"]
m_prophet = evaluate(y_true, result["yhat"])
m_naive   = evaluate(y_true, np.full(len(y_true), train["y"].iloc[-1]))

metrics = pd.DataFrame({"Prophet": m_prophet, "Naive (ราคาล่าสุด)": m_naive}).round(2)
print(metrics)
mae, rmse, mape, acc = m_prophet["MAE"], m_prophet["RMSE"], m_prophet["MAPE (%)"], m_prophet["Accuracy (%)"]
print(f"\n>>> ความแม่นยำของโมเดล (Accuracy) = {acc:.2f}%")

inside = ((y_true >= result["yhat_lower"]) & (y_true <= result["yhat_upper"])).mean() * 100
print(f"\nราคาจริงอยู่ในช่วงพยากรณ์ (yhat_lower–yhat_upper): {inside:.1f}% ของวัน")

In [ ]:
plt.plot(train["ds"].iloc[-500:], train["y"].iloc[-500:], label="Train", color="gray")
plt.plot(result["ds"], result["y"], label="Actual (Test)", color="black")
plt.plot(result["ds"], result["yhat"], label="Predicted", color="tab:blue")
plt.fill_between(result["ds"], result["yhat_lower"], result["yhat_upper"], alpha=.2, color="tab:blue")
plt.title(f"Test: MAE={mae:.2f}  RMSE={rmse:.2f}  MAPE={mape:.2f}%  Accuracy={acc:.2f}%")
plt.legend(); plt.grid(alpha=.3); plt.show()

## 9.1 Cross-validation หลายช่วงเวลา (ผลที่น่าเชื่อถือกว่า)
การทดสอบช่วงเดียว (ข้อ 9) อาจฟลุ๊ค ตรงนี้จะย้อนไปเทรนและทดสอบซ้ำ **ทุก ๆ 30 วันตลอด 4 ปีที่ผ่านมา** (~48 รอบ) แต่ละรอบพยากรณ์ล่วงหน้า `HORIZON` วัน แล้วเฉลี่ยค่า MAE, RMSE, MAPE

ใช้เวลารันประมาณ 1–3 นาที

In [ ]:
CV_YEARS, CV_STEP = 4, 30
cutoffs = pd.date_range(data["ds"].max() - pd.Timedelta(days=365 * CV_YEARS),
                        data["ds"].max() - pd.Timedelta(days=TEST_DAYS), freq=f"{CV_STEP}D")
rows = []
for cut in cutoffs:
    tr = data[data["ds"] <= cut]
    te = data[(data["ds"] > cut) & (data["ds"] <= cut + pd.Timedelta(days=TEST_DAYS))]
    m = fit_model(tr)
    p = predict_price(m, te[["ds"]])["yhat"].values
    r_p = evaluate(te["y"], p)
    r_n = evaluate(te["y"], np.full(len(te), tr["y"].iloc[-1]))
    rows.append({"cutoff": cut.date(), **{f"Prophet {k}": v for k, v in r_p.items()},
                 **{f"Naive {k}": v for k, v in r_n.items()}})
    print(f"{cut.date()}  Prophet MAPE {r_p['MAPE (%)']:6.2f}%  Accuracy {r_p['Accuracy (%)']:6.2f}%   |  Naive MAPE {r_n['MAPE (%)']:6.2f}%")

cv = pd.DataFrame(rows)
cv_summary = pd.DataFrame({
    "Prophet": [cv[f"Prophet {k}"].mean() for k in ["MAE", "RMSE", "MAPE (%)", "Accuracy (%)"]],
    "Naive (ราคาล่าสุด)": [cv[f"Naive {k}"].mean() for k in ["MAE", "RMSE", "MAPE (%)", "Accuracy (%)"]],
}, index=["MAE", "RMSE", "MAPE (%)", "Accuracy (%)"]).round(2)
print(f"\nค่าเฉลี่ยจาก {len(cv)} รอบ (พยากรณ์ล่วงหน้า {TEST_DAYS} วัน):")
print(cv_summary)
print(f"\n>>> ความแม่นยำเฉลี่ย (Accuracy) ของ Prophet = {cv_summary.loc['Accuracy (%)', 'Prophet']:.2f}%")

## 10. เทรนใหม่ด้วยข้อมูลทั้งหมด แล้วพยากรณ์อนาคต
หลังวัดผลแล้ว เทรนใหม่ด้วยข้อมูลครบ 15 ปี เพื่อให้โมเดลรู้ราคาล่าสุด

In [ ]:
final_model = fit_model(data)

future = final_model.make_future_dataframe(periods=FUTURE_DAYS, freq="B")   # B = เฉพาะวันทำการ
forecast = predict_price(final_model, future)
forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail(10)

In [ ]:
hist = data[data["ds"] >= data["ds"].max() - pd.Timedelta(days=730)]
fut  = forecast[forecast["ds"] > data["ds"].max()]
plt.plot(hist["ds"], hist["y"], color="black", label="Actual")
plt.plot(fut["ds"], fut["yhat"], color="tab:blue", label="Forecast")
plt.fill_between(fut["ds"], fut["yhat_lower"], fut["yhat_upper"], alpha=.2, color="tab:blue")
plt.title(f"Forecast next {FUTURE_DAYS} business days"); plt.legend(); plt.grid(alpha=.3); plt.show()

# องค์ประกอบของโมเดล (เทรนด์ / ฤดูกาลรายปี) — แกน y เป็นสเกล log
final_model.plot_components(final_model.predict(future)); plt.show()

## 11. เซฟโมเดลและผลพยากรณ์

In [ ]:
with open(MODEL_PATH, "w") as f:
    f.write(model_to_json(final_model))   # หมายเหตุ: โมเดลเทรนบน log(ราคา) ตอนใช้ต้องใช้ predict_price() หรือ np.exp(yhat)

future_only = forecast[forecast["ds"] > data["ds"].max()][["ds", "yhat", "yhat_lower", "yhat_upper"]]
future_only.round({"yhat": 2, "yhat_lower": 2, "yhat_upper": 2}).to_csv(FORECAST_PATH, index=False)

metrics.to_csv(os.path.join(MODEL_DIR, "gold_prophet_metrics_test.csv"))
cv_summary.to_csv(os.path.join(MODEL_DIR, "gold_prophet_metrics_cv.csv"))

print("เซฟโมเดล    ->", MODEL_PATH)
print("เซฟพยากรณ์  ->", FORECAST_PATH)
print("เซฟค่าความแม่นยำ -> MODEL/gold_prophet_metrics_test.csv, MODEL/gold_prophet_metrics_cv.csv")

## 12. ทดสอบโหลดโมเดลกลับมาใช้

In [ ]:
with open(MODEL_PATH) as f:
    loaded = model_from_json(f.read())

check = predict_price(loaded, loaded.make_future_dataframe(periods=5, freq="B"))
check[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail(5)